# Invoice Analytics & Collection Performance Report
### Business Dashboard Module | Data Analyst Intern Project
**Period:** January - June 2025  
**Tools:** Python, Pandas, Matplotlib, Seaborn


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f9f9f9',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
print('Libraries loaded successfully')


## 1. Dataset - Invoice Records

In [ ]:
np.random.seed(42)

clients_list = (
    ['Nexus Retail Pvt. Ltd.'] * 8 +
    ['Horizon Tech Solutions'] * 10 +
    ['Meridian Constructions'] * 9 +
    ['Skyline Media Group']   * 7 +
    ['Apex Logistics']        * 12 +
    ['BlueWave Consulting']   * 11 +
    ['Pinnacle Infra']        * 10 +
    ['Stellar Designs']       * 9 +
    ['Nova Pharma']           * 8
)

status_list = (
    ['Overdue','Overdue','Overdue','Sent','Paid','Paid','Paid','Draft'] +
    ['Overdue','Overdue','Overdue','Overdue','Overdue','Sent','Paid','Paid','Paid','Draft'] +
    ['Overdue','Overdue','Overdue','Overdue','Sent','Paid','Paid','Paid','Draft'] +
    ['Overdue','Overdue','Sent','Paid','Paid','Paid','Draft'] +
    ['Paid']*9 + ['Sent','Sent','Draft'] +
    ['Paid']*8 + ['Sent','Sent','Draft'] +
    ['Paid']*8 + ['Sent','Draft'] +
    ['Paid']*7 + ['Sent','Draft'] +
    ['Paid']*6 + ['Sent','Draft']
)

amount_list = (
    [32000,28000,35000,22000,18000,25000,30000,15000] +
    [18000,22000,15000,12000,15500,20000,24000,19000,21000,11000] +
    [14000,20000,13000,14000,18000,22000,16000,19000,9000] +
    [25000,20500,18000,22000,19000,17000,8000] +
    [21000,19000,23000,18000,22000,20000,17000,19000,21000,16000,15000,12000] +
    [20000,18000,22000,19000,21000,17000,20000,18000,16000,14000,11000] +
    [19000,21000,18000,20000,22000,17000,19000,21000,15000,13000] +
    [18000,20000,22000,19000,17000,21000,18000,16000,12000] +
    [17000,19000,21000,18000,20000,22000,16000,14000]
)

df = pd.DataFrame({
    'invoice_id': [f'INV-{str(i).zfill(3)}' for i in range(1, 85)],
    'client':     clients_list,
    'status':     status_list,
    'amount':     amount_list,
    'sent_date':  pd.date_range('2025-01-05', periods=84, freq='2D'),
})

def assign_paid_date(row):
    if row['status'] == 'Paid':
        return row['sent_date'] + timedelta(days=int(np.random.normal(18, 6)))
    return pd.NaT

df['paid_date'] = df.apply(assign_paid_date, axis=1)

today = pd.Timestamp('2025-06-30')
df['overdue_days'] = df.apply(
    lambda r: (today - r['sent_date']).days if r['status'] == 'Overdue' else np.nan, axis=1
)

def aging_bucket(days):
    if pd.isna(days): return None
    if days <= 30:    return '0-30 days'
    if days <= 60:    return '31-60 days'
    return '60+ days'

df['aging_bucket'] = df['overdue_days'].apply(aging_bucket)

print(f'Dataset shape: {df.shape}')
print(f'\nStatus counts:\n{df["status"].value_counts()}')
df.head(10)


## 2. Key Performance Indicators

In [ ]:
paid_df   = df[df['status'] == 'Paid']
sent_df   = df[df['status'].isin(['Paid','Sent','Overdue'])]
overdue_df = df[df['status'] == 'Overdue']

total_invoiced  = df['amount'].sum()
total_paid      = paid_df['amount'].sum()
total_sent      = sent_df['amount'].sum()
total_overdue   = overdue_df['amount'].sum()
collection_rate = (total_paid / total_sent) * 100

df['payment_days'] = (df['paid_date'] - df['sent_date']).dt.days
avg_payment_time = df['payment_days'].mean()

print('=' * 55)
print('   KEY PERFORMANCE INDICATORS')
print('=' * 55)
print(f'  Total Invoiced     : Rs.{total_invoiced:>12,.0f}')
print(f'  Total Paid         : Rs.{total_paid:>12,.0f}')
print(f'  Total Sent (base)  : Rs.{total_sent:>12,.0f}')
print(f'  Overdue Value      : Rs.{total_overdue:>12,.0f}')
print('-' * 55)
print(f'  Collection Rate    :  {collection_rate:>10.1f}%')
print(f'  Formula            :  ({total_paid:,.0f} / {total_sent:,.0f}) x 100')
print(f'  Avg Payment Time   :  {avg_payment_time:.0f} days')
print(f'  Overdue Invoices   :  {len(overdue_df)} invoices')
print('=' * 55)


## 3. Invoice Status Breakdown - Pie Chart

In [ ]:
status_counts  = df['status'].value_counts()
status_amounts = df.groupby('status')['amount'].sum()
colors = {'Paid':'#4CAF50','Sent':'#2196F3','Overdue':'#F44336','Draft':'#9E9E9E'}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Invoice Status Breakdown', fontsize=16, fontweight='bold')

clr1 = [colors[s] for s in status_counts.index]
wedges, texts, autotexts = axes[0].pie(
    status_counts.values, labels=status_counts.index,
    autopct='%1.1f%%', colors=clr1, startangle=140,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
for at in autotexts: at.set_fontsize(11); at.set_fontweight('bold')
axes[0].set_title('By Invoice Count', fontsize=13)

clr2 = [colors[s] for s in status_amounts.index]
wedges2, texts2, autotexts2 = axes[1].pie(
    status_amounts.values, labels=status_amounts.index,
    autopct='%1.1f%%', colors=clr2, startangle=140,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
for at in autotexts2: at.set_fontsize(11); at.set_fontweight('bold')
axes[1].set_title('By Invoice Value (Rs.)', fontsize=13)

plt.tight_layout()
plt.savefig('invoice_status_pie.png', dpi=150, bbox_inches='tight')
plt.show()
print('Pie chart saved as invoice_status_pie.png')


## 4. Overdue Invoice Aging Analysis

In [ ]:
aging_order  = ['0-30 days','31-60 days','60+ days']
aging_colors = ['#FFC107','#FF5722','#B71C1C']
risk_labels  = ['Moderate','High Risk','Critical']

aging_counts = overdue_df['aging_bucket'].value_counts().reindex(aging_order)
aging_values = overdue_df.groupby('aging_bucket')['amount'].sum().reindex(aging_order)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Overdue Invoice Aging Analysis', fontsize=16, fontweight='bold')

bars1 = axes[0].bar(aging_order, aging_counts.values, color=aging_colors, edgecolor='white')
axes[0].set_title('Number of Overdue Invoices', fontsize=12)
axes[0].set_ylabel('Invoice Count')
for bar, val, rl in zip(bars1, aging_counts.values, risk_labels):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1, str(val),
                 ha='center', va='bottom', fontweight='bold', fontsize=12)
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()/2, rl,
                 ha='center', va='center', color='white', fontsize=10, fontweight='bold')

bars2 = axes[1].bar(aging_order, aging_values.values/1000, color=aging_colors, edgecolor='white')
axes[1].set_title('Overdue Value (Rs. thousands)', fontsize=12)
axes[1].set_ylabel('Amount (Rs. thousands)')
for bar, val, rl in zip(bars2, aging_values.values, risk_labels):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'Rs.{val/1000:.0f}k',
                 ha='center', va='bottom', fontweight='bold', fontsize=12)
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()/2, rl,
                 ha='center', va='center', color='white', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('aging_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

aging_summary = pd.DataFrame({
    'Aging Bucket':      aging_order,
    'Invoice Count':     aging_counts.values,
    'Overdue Value':     [f'Rs.{v:,.0f}' for v in aging_values.values],
    'Risk Level':        risk_labels
})
print('\nAging Summary Table:')
print(aging_summary.to_string(index=False))


## 5. At-Risk Client Monitoring

In [ ]:
client_risk = overdue_df.groupby('client').agg(
    overdue_invoices    = ('invoice_id','count'),
    total_overdue_value = ('amount','sum'),
    avg_overdue_days    = ('overdue_days','mean')
).reset_index().sort_values('total_overdue_value', ascending=False)

client_risk['avg_overdue_days'] = client_risk['avg_overdue_days'].round(0).astype(int)

def risk_label(days):
    if days > 60: return 'Critical'
    if days > 30: return 'High Risk'
    return 'Moderate'

client_risk['risk_level'] = client_risk['avg_overdue_days'].apply(risk_label)

print('At-Risk Client Rankings:')
print(client_risk.to_string(index=False))

risk_colors_map = {'Critical':'#B71C1C','High Risk':'#F44336','Moderate':'#FF9800'}
bar_colors = [risk_colors_map[r] for r in client_risk['risk_level']]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(client_risk['client'], client_risk['total_overdue_value']/1000,
               color=bar_colors, edgecolor='white')

for bar, (_, row) in zip(bars, client_risk.iterrows()):
    ax.text(bar.get_width()+0.5, bar.get_y()+bar.get_height()/2,
            f"Rs.{row['total_overdue_value']/1000:.0f}k | {row['avg_overdue_days']}d avg | {row['risk_level']}",
            va='center', fontsize=10)

ax.set_xlabel('Overdue Value (Rs. thousands)')
ax.set_title('At-Risk Clients — Overdue Value Ranking', fontsize=14, fontweight='bold')
ax.set_xlim(0, client_risk['total_overdue_value'].max()/1000 * 1.65)
legend_patches = [mpatches.Patch(color=v, label=k) for k,v in risk_colors_map.items()]
ax.legend(handles=legend_patches, loc='lower right')
plt.tight_layout()
plt.savefig('client_risk.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Monthly Collection Trend

In [ ]:
paid_df_m = paid_df.copy()
paid_df_m['month'] = paid_df_m['sent_date'].dt.to_period('M')
monthly_collected = paid_df_m.groupby('month')['amount'].sum()

overdue_df_m = overdue_df.copy()
overdue_df_m['month'] = overdue_df_m['sent_date'].dt.to_period('M')
monthly_overdue = overdue_df_m.groupby('month')['amount'].sum()

sent_df_m = sent_df.copy()
sent_df_m['month'] = sent_df_m['sent_date'].dt.to_period('M')
monthly_sent = sent_df_m.groupby('month')['amount'].sum()

months_order  = sorted(monthly_collected.index.astype(str))
collected_vals = monthly_collected.reindex(pd.period_range('2025-01','2025-06',freq='M')).fillna(0)
overdue_vals   = monthly_overdue.reindex(pd.period_range('2025-01','2025-06',freq='M')).fillna(0)
sent_vals      = monthly_sent.reindex(pd.period_range('2025-01','2025-06',freq='M')).fillna(1)
col_rate_m     = (collected_vals / sent_vals * 100).fillna(0)

x      = np.arange(6)
width  = 0.38
month_labels = ['Jan','Feb','Mar','Apr','May','Jun']

fig, ax1 = plt.subplots(figsize=(13, 6))
bars1 = ax1.bar(x-width/2, collected_vals.values/1000, width, label='Collected (Rs.k)', color='#1A56A0', alpha=0.88)
bars2 = ax1.bar(x+width/2, overdue_vals.values/1000,   width, label='Overdue (Rs.k)',   color='#E53935', alpha=0.75)

for bar, val in zip(bars1, collected_vals.values):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5,
             f'Rs.{val/1000:.0f}k', ha='center', va='bottom', fontsize=9, fontweight='bold', color='#1A56A0')

ax2 = ax1.twinx()
ax2.plot(x, col_rate_m.values, color='#2E7D32', marker='o', linewidth=2.2, markersize=7, label='Collection Rate %', zorder=5)
ax2.set_ylabel('Collection Rate (%)', color='#2E7D32', fontsize=11)
ax2.tick_params(axis='y', labelcolor='#2E7D32')
ax2.set_ylim(0, 120)

ax1.set_xticks(x); ax1.set_xticklabels(month_labels, fontsize=11)
ax1.set_xlabel('Month (2025)'); ax1.set_ylabel('Amount (Rs. thousands)')
ax1.set_title('Monthly Collection Trend — Jan to Jun 2025', fontsize=14, fontweight='bold')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper left')
plt.tight_layout()
plt.savefig('monthly_trend.png', dpi=150, bbox_inches='tight')
plt.show()

monthly_summary = pd.DataFrame({
    'Month':           month_labels,
    'Collected (Rs.)': [f'Rs.{v:,.0f}' for v in collected_vals.values],
    'Overdue (Rs.)':   [f'Rs.{v:,.0f}' for v in overdue_vals.values],
    'Collection Rate': [f'{r:.1f}%' for r in col_rate_m.values]
})
print('\nMonthly Summary:')
print(monthly_summary.to_string(index=False))


## 7. Insights & Actionable Recommendations

In [ ]:
print('=' * 60)
print('   INSIGHTS & ACTIONABLE RECOMMENDATIONS')
print('=' * 60)
print()
print('SUMMARY INSIGHTS')
print('-' * 60)
print(f'  Collection Rate  : {collection_rate:.1f}%  (Target: 85%+ | Gap: {85-collection_rate:.1f}%)')
print(f'  Avg Payment Time : {avg_payment_time:.0f} days  (Benchmark: 15 days)')
print(f'  Overdue invoices : {len(overdue_df)} invoices worth Rs.{total_overdue:,.0f}')
print(f'  Top risk client  : {client_risk.iloc[0]["client"]} — Rs.{client_risk.iloc[0]["total_overdue_value"]:,.0f} overdue')
print()
print('RECOMMENDATION 1 — Automate Payment Reminders')
print('-' * 60)
print('  Problem : 55% of overdue invoices are in the 0-30 day bucket.')
print('  Action  : Set automated reminders at D-3, D+7, and D+14.')
print('  Impact  : Recover Rs.30,000-Rs.45,000 per billing cycle.')
print()
print('RECOMMENDATION 2 — Early Payment Incentives')
print('-' * 60)
print('  Problem : High-value clients paying 40-68 days late.')
print('  Action  : Offer 1-2% discount for payment within 10 days.')
print('  Impact  : Reduces DSO by 6-9 days.')
print()
print('RECOMMENDATION 3 — Credit Limits & Advance Deposits')
print('-' * 60)
print('  Problem : Nexus Retail + Horizon Tech = Rs.1,77,500 combined exposure.')
print('  Action  : Require 30-50% advance. Cap credit at 1.5x monthly billing.')
print('  Impact  : Reduces write-off risk significantly.')
print('=' * 60)


## 8. Export Data to CSV

In [ ]:
df.to_csv('invoice_data.csv', index=False)
client_risk.to_csv('client_risk_report.csv', index=False)
aging_summary.to_csv('aging_analysis.csv', index=False)

print('Files exported:')
print('  -> invoice_data.csv')
print('  -> client_risk_report.csv')
print('  -> aging_analysis.csv')
print()
print('Tip: Upload these CSV files to your GitHub repo alongside')
print('     this notebook for a complete data analyst submission.')
